# MCP: el protocolo de las integraciones

**Lección 1 · Clase 5.4** — antes de conectar un LLM con *algo* (una base de datos, un sistema de ventas, un pronóstico del tiempo), hay que decidir **cómo** se conecta. Durante 2023 y 2024 cada framework inventó su propio formato de herramientas: las tools de OpenAI no eran las de Anthropic, ni las de LangChain, ni las del framework de turno. Integrar N sistemas con M agentes costaba N×M adaptadores.

**MCP (Model Context Protocol)** ataca exactamente ese problema: es un protocolo abierto para exponer capacidades a los LLMs. El servidor las declara una vez, con un esquema que cualquier cliente entiende, y da lo mismo si del otro lado hay Claude Desktop, un agente de LangChain o tu propia aplicación. Por eso la analogía que pegó: *el USB-C de las integraciones*.

Un servidor MCP expone tres tipos de bloques, y la diferencia importante entre ellos es **quién decide usarlos**:

| Bloque | Para qué sirve | Quién lo controla | Ejemplo |
|---|---|---|---|
| **Tools** | Acciones | El **modelo** decide llamarlas | Crear un pedido, consultar un pronóstico |
| **Resources** | Datos de contexto | La **aplicación** decide leerlos | Un catálogo, un documento, condiciones actuales |
| **Prompts** | Plantillas de interacción | El **usuario** decide invocarlas | "Planear un viaje", "Asistente de compra" |

| | |
|---|---|
| **Anatomía** | Un servidor FastMCP en 15 líneas. |
| **Dos servidores reales** | Transacciones y pronóstico de nieve de Valle Nevado, con salida estructurada, logging y progreso. |
| **El cliente** | Descubrir tools, resources y prompts sin conocer el servidor de antemano. |
| **El agente** | `create_agent` conectado a los dos servidores — y el contraste medido: la misma pregunta con y sin MCP. |
| **Transportes** | streamable-http vs stdio, y cuándo usar cada uno. |


In [ ]:
# Esta lección usa el entorno uv del README. Si la corres en Colab, descomenta:
# %pip install -q mcp==1.29.0 langchain-mcp-adapters==0.3.2 langchain==1.3.14 \
#   langchain-core==1.5.3 langchain-openai==1.4.2 python-dotenv==1.2.2
from dotenv import load_dotenv
import os

load_dotenv()

try:
    from google.colab import userdata  # type: ignore
    try:
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY") or os.environ.get("OPENAI_API_KEY", "")
    except Exception:
        pass
except Exception:
    pass

# Los clientes MCP y httpx loguean cada request a nivel INFO: silencio, que es una clase.
import logging
import warnings

for ruidoso in ("httpx", "mcp"):
    logging.getLogger(ruidoso).setLevel(logging.WARNING)
warnings.filterwarnings("ignore", message=r"Field 'lifespan' has an incomplete definition")

HAY_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
print("OPENAI_API_KEY presente:", HAY_OPENAI)
if not HAY_OPENAI:
    print("⚠️ Sin OPENAI_API_KEY los servidores y el cliente MCP funcionan igual,")
    print("   pero la sección del agente se salta.")

# En Colab los .py de los servidores no existen: los bajamos del repo público.
import urllib.request
from pathlib import Path

BASE_RAW = (
    "https://raw.githubusercontent.com/josepenam/clases-diplomado-gen-ia/main/"
    "class_5_4_integraciones/leccion1_mcp/"
)


def asegurar(nombre: str) -> Path:
    """Devuelve la ruta local del archivo; en Colab lo baja del repo público."""
    ruta = Path(nombre)
    if not ruta.exists():
        pedido = urllib.request.Request(
            BASE_RAW + nombre, headers={"User-Agent": "Mozilla/5.0 (clase-diplomado-gen-ia)"}
        )
        ruta.write_bytes(urllib.request.urlopen(pedido).read())
    return ruta


for archivo in ("servidor_transacciones.py", "servidor_pronostico_nieve.py"):
    asegurar(archivo)
    print("✓", archivo)


## Anatomía: un servidor MCP en 15 líneas

`FastMCP` (parte del SDK oficial de Python) registra capacidades con decoradores, y **genera los esquemas a partir de las anotaciones de tipo**. Eso es lo que hace al protocolo "LLM-friendly": el cliente no recibe código, recibe una descripción estructurada — nombre, parámetros, tipos, docstring — que el modelo puede leer para decidir qué llamar y con qué argumentos.

Definimos el servidor de juguete acá mismo para ver las piezas. **No lo vamos a levantar desde esta celda**: `mcp.run()` bloquea el proceso que lo ejecuta (es un servidor HTTP), y ese proceso sería el kernel del notebook. Los servidores de verdad viven en sus propios archivos y corren como procesos aparte — eso viene en la sección siguiente.

In [ ]:
from mcp.server.fastmcp import FastMCP

demo = FastMCP("Matematicas")


@demo.tool()
def sumar(a: int, b: int) -> int:
    """Suma dos números enteros."""
    return a + b


@demo.tool()
def multiplicar(a: int, b: int) -> int:
    """Multiplica dos números enteros."""
    return a * b


@demo.resource("saludo://{nombre}")
def saludo(nombre: str) -> str:
    """Devuelve un saludo para el nombre dado."""
    return f"¡Hola, {nombre}! Bienvenido a MCP."


# Lo que el cliente vería: los esquemas generados desde los tipos y docstrings.
# (await de primer nivel: en Jupyter el event loop ya está corriendo)
for herramienta in await demo.list_tools():
    print(f"tool: {herramienta.name}{herramienta.inputSchema.get('required', [])} — {herramienta.description}")

# Para servirlo de verdad sería:  demo.run(transport="streamable-http")
# (bloquea el proceso: por eso los servidores reales van en archivos aparte)

## Dos servidores reales: Valle Nevado

Para el resto de la lección usamos dos servidores mock que simulan sistemas de un centro de esquí — el escenario que nos va a acompañar toda la clase:

- **[`servidor_transacciones.py`](servidor_transacciones.py)** — el sistema de ventas: tools `crear_pedido` y `aplicar_descuento`, el resource `resort://catalogo` y el prompt `asistente_compra`. Las tools devuelven **salida estructurada**: modelos Pydantic (`Pedido`, `ItemCarrito`) que MCP convierte en esquema JSON, así el cliente recibe datos tipados y no texto suelto.
- **[`servidor_pronostico_nieve.py`](servidor_pronostico_nieve.py)** — meteorología: tools `obtener_pronostico` e `indice_riesgo_aludes`, un resource estático (`nieve://estaciones`), uno **dinámico por plantilla de URI** (`nieve://condiciones/{estacion}`) y el prompt `plan_viaje`.

`obtener_pronostico` muestra además dos capacidades del protocolo que van más allá de "función con esquema": recibe un `ctx: Context` que FastMCP inyecta, y por él manda **logs** (`ctx.info`) y **progreso** (`ctx.report_progress`) *al cliente*, dentro de la misma sesión MCP. En una herramienta que tarda —una consulta pesada, un scraping— eso es lo que permite mostrarle algo al usuario mientras espera.

### Levantarlos desde el notebook

Cada servidor corre como **proceso aparte** (así funcionan en la vida real: el agente no importa tu código, se conecta por red). Los lanzamos con `subprocess.Popen`, esperamos a que el puerto responda, y guardamos los procesos en una lista para matarlos al final — la última celda del notebook hace esa limpieza.

In [ ]:
import atexit
import socket
import subprocess
import sys
import time

PUERTO_TRANSACCIONES = 8801
PUERTO_PRONOSTICO = 8802

PROCESOS: list[subprocess.Popen] = []


def esperar_puerto(puerto: int, timeout: float = 20.0) -> bool:
    """Espera a que un puerto local acepte conexiones."""
    limite = time.time() + timeout
    while time.time() < limite:
        with socket.socket() as s:
            s.settimeout(0.5)
            if s.connect_ex(("127.0.0.1", puerto)) == 0:
                return True
        time.sleep(0.3)
    return False


def lanzar_servidor(script: str, variable: str, puerto: int) -> None:
    proceso = subprocess.Popen(
        [sys.executable, script],
        env={**os.environ, variable: str(puerto)},
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    PROCESOS.append(proceso)
    if not esperar_puerto(puerto):
        raise RuntimeError(f"{script} no levantó en el puerto {puerto}")
    print(f"✓ {script} corriendo en http://localhost:{puerto}/mcp")


def terminar_servidores() -> None:
    for proceso in PROCESOS:
        if proceso.poll() is None:
            proceso.terminate()
    for proceso in PROCESOS:
        try:
            proceso.wait(timeout=5)
        except subprocess.TimeoutExpired:
            proceso.kill()
    PROCESOS.clear()


atexit.register(terminar_servidores)  # red de seguridad si el kernel muere

lanzar_servidor("servidor_transacciones.py", "MCP_PORT_TRANSACCIONES", PUERTO_TRANSACCIONES)
lanzar_servidor("servidor_pronostico_nieve.py", "MCP_PORT_PRONOSTICO", PUERTO_PRONOSTICO)

> **Alternativa interactiva:** el SDK trae un inspector visual. En una terminal, `uv run mcp dev servidor_pronostico_nieve.py` levanta una UI en `http://127.0.0.1:6274` donde puedes explorar e invocar tools, resources y prompts a mano — útil para depurar un servidor antes de conectarle un agente.

## El cliente: descubrir capacidades

Acá está la gracia del protocolo. El cliente **no sabe nada** de los servidores — solo tiene sus URLs — y aun así puede preguntarles qué ofrecen. `MultiServerMCPClient` (de `langchain-mcp-adapters`) se conecta a los dos y convierte las tools MCP en tools de LangChain listas para un agente.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

cliente = MultiServerMCPClient(
    {
        "transacciones": {
            "transport": "streamable_http",
            "url": f"http://localhost:{PUERTO_TRANSACCIONES}/mcp",
        },
        "pronostico": {
            "transport": "streamable_http",
            "url": f"http://localhost:{PUERTO_PRONOSTICO}/mcp",
        },
    }
)

herramientas = await cliente.get_tools()

print(f"{len(herramientas)} tools descubiertas:\n")
for herramienta in herramientas:
    print(f"  {herramienta.name:<22} {herramienta.description}")

# El esquema que ve el modelo — generado desde los tipos del servidor:
import json

ejemplo = next(h for h in herramientas if h.name == "indice_riesgo_aludes")
print("\nEsquema de indice_riesgo_aludes:")
print(json.dumps(ejemplo.args, indent=2, ensure_ascii=False))

Las tools son la mitad de la historia. Los **resources** y **prompts** también se descubren por protocolo — con la diferencia de control que vimos en la tabla inicial: no los llama el modelo, los lee la aplicación (o los invoca el usuario).

In [ ]:
from langchain_mcp_adapters.prompts import load_mcp_prompt
from langchain_mcp_adapters.resources import load_mcp_resources

async with cliente.session("pronostico") as sesion:
    # Resource estático
    [estaciones] = await load_mcp_resources(sesion, uris=["nieve://estaciones"])
    print("nieve://estaciones →", estaciones.data)

    # Resource dinámico: el parámetro va en el path de la URI
    [condiciones] = await load_mcp_resources(sesion, uris=["nieve://condiciones/La Parva"])
    print("\nnieve://condiciones/La Parva →", condiciones.data)

    # Prompt: plantilla parametrizada que devuelve mensajes listos para el modelo.
    # Ojo: por protocolo los argumentos de un prompt viajan como strings
    # (el servidor los convierte a sus tipos declarados).
    mensajes = await load_mcp_prompt(sesion, "plan_viaje", arguments={"estacion": "Portillo", "dias": "3"})
    print("\nprompt plan_viaje →", mensajes[0].content)

## El agente conectado

Ahora el experimento que importa. Armamos **el mismo agente dos veces** — mismo modelo, mismo prompt — y la única diferencia es si tiene o no las herramientas MCP. La pregunta necesita datos que el modelo no puede saber: el pronóstico de nieve de mañana.

(El agente es `create_agent` de LangChain 1.x, el mismo de la clase 5.3: devuelve un grafo que se invoca con `{"messages": [...]}`.)

In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

MODELO = "gpt-5-mini"

INSTRUCCIONES = (
    "Eres el asistente de Valle Nevado. Responde en español, breve y concreto. "
    "Usa las herramientas disponibles cuando la pregunta requiera datos; "
    "si no tienes cómo saber algo, dilo honestamente."
)

PREGUNTA = "¿Cuánta nieve va a caer mañana en La Parva y qué riesgo de aludes hay?"


def contar_tools(resultado) -> int:
    return sum(1 for m in resultado["messages"] if type(m).__name__ == "ToolMessage")


if HAY_OPENAI:
    modelo = ChatOpenAI(model=MODELO)

    agente_solo = create_agent(modelo, tools=[], system_prompt=INSTRUCCIONES)
    agente_mcp = create_agent(modelo, tools=herramientas, system_prompt=INSTRUCCIONES)

    for nombre, agente in [("SIN herramientas", agente_solo), ("CON MCP", agente_mcp)]:
        resultado = await agente.ainvoke({"messages": [{"role": "user", "content": PREGUNTA}]})
        print(f"═══ Agente {nombre} ═══")
        print(f"llamadas a tools: {contar_tools(resultado)}")
        print(resultado["messages"][-1].content)
        print()
else:
    print("⛔ Falta OPENAI_API_KEY: esta sección necesita el modelo.")

El contraste es el punto de la lección: **sin la integración el agente solo puede declararse ignorante** (o peor, inventar); con MCP llama a `obtener_pronostico` y responde con datos. Y no tuvimos que escribir ni un adaptador: las tools llegaron solas por el protocolo.

Probemos las consultas que mezclan los dos servidores — el agente decide solo a cuál llamar:

In [ ]:
async def consultar(pregunta: str) -> None:
    print(f"🙋 {pregunta}")
    resultado = await agente_mcp.ainvoke({"messages": [{"role": "user", "content": pregunta}]})
    tools_usadas = [
        llamada["name"]
        for m in resultado["messages"]
        if getattr(m, "tool_calls", None)
        for llamada in m.tool_calls
    ]
    print(f"🔧 tools: {tools_usadas}")
    print(f"🤖 {resultado['messages'][-1].content}\n")


if HAY_OPENAI:
    await consultar(
        "Ayer nevó 60 cm en La Parva con vientos de 70 km/h. "
        "¿Cómo está el riesgo de aludes en Barros Negros (38 grados de inclinación)?"
    )
    await consultar("Quiero comprar 3 forfaits para Valle Nevado a 65.000 CLP cada uno, con un 10% de descuento.")
else:
    print("⛔ Falta OPENAI_API_KEY.")

## Transportes: streamable-http vs stdio

Usamos **streamable-http** porque nuestros servidores corren como servicios independientes con URL — el caso de un servidor compartido por el equipo o publicado en la red interna. La alternativa, **stdio**, hace que el *cliente* lance el servidor como subproceso y le hable por stdin/stdout: es lo que usan Claude Desktop y los IDEs para servidores locales (no hay puerto, no hay red, muere con el cliente).

Con `langchain-mcp-adapters` el cambio es solo de configuración:

```python
cliente_stdio = MultiServerMCPClient(
    {
        "pronostico": {
            "transport": "stdio",
            "command": sys.executable,
            "args": ["servidor_pronostico_nieve.py"],
        },
    }
)
```

La regla práctica: **stdio para herramientas locales de un solo usuario; streamable-http para servicios compartidos** (y ahí aparecen las preocupaciones de siempre: autenticación, autorización, rate limits).

## La frontera

Lo que no cubrimos, para saber que existe:

- **Elicitación** (`ctx.elicit`): el servidor puede pedirle datos adicionales al usuario a mitad de una tool ("¿confirmas la compra?"). Requiere que el *cliente* lo soporte — `langchain-mcp-adapters` todavía no lo implementa; Claude Desktop y otros clientes sí.
- **Sampling**: el servidor puede pedirle al cliente que ejecute una llamada al LLM por él — útil para servidores que necesitan razonar sin tener su propia llave.
- **OAuth para servidores remotos**: el estándar define cómo autenticarse contra servidores MCP de terceros (GitHub, Linear, etc.) sin repartir API keys.
- **MCP SDK v2**: ya está publicado (con una API nueva, `MCPServer`), pero los adaptadores de LangChain aún requieren la serie 1.x — por eso esta lección pinnea `mcp==1.29.0`. El protocolo es el mismo; cambia la librería.

## Limpieza

Los servidores son procesos reales: hay que terminarlos. (Si se te olvida, el `atexit` que registramos lo intenta al morir el kernel.)

In [ ]:
terminar_servidores()
print("Servidores MCP terminados.")